In [1]:
import pandas as pd
import os
from utils.dataframe_tools import global_stratified_split, global_stratified_split_memory_optimized
from utils.dataframe_tools import truncate_to_block_by_schema, augment_main_block, create_specialist_dataset, create_specialist_dataset_intersect, augment_main_block_v2
from utils.pruning_and_merge import merge_field_blocks_tree_similarity
from utils.dataframe_tools import truncate_to_block_by_schema, augment_main_block_v2, augment_main_block_top_k

In [2]:
# ori_csv_path = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'completeness', 'dataset_29_completed_label.csv')
# output_dir_train_test = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'train_test')
# output_dir_blocks_first = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'train_test', 'blocks', 'first_truncation')
# output_dir_blocks_second = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'train_test', 'blocks', 'merge')
# ori_csv_path = os.path.join('..', 'TrafficData', 'dataset_20_d2_csv', 'dataset_20_d2.csv')
# output_dir_train_test = os.path.join('..', 'TrafficData', 'dataset_20_d2_csv_merged', 'train_test')
# output_dir_blocks_first = os.path.join('..', 'TrafficData', 'dataset_20_d2_csv_merged', 'train_test', 'blocks', 'first_truncation')
# output_dir_blocks_second = os.path.join('..', 'TrafficData', 'dataset_20_d2_csv_merged', 'train_test', 'blocks', 'merge')

ori_csv_path = os.path.join('..', 'TrafficData', 'datasets_consolidate', 'dataset_20_d2.csv')
output_dir_train_test = os.path.join('..', 'TrafficData', 'datasets_split', 'dataset_20_d2')
# E:\Coding\TrafficData\datasets_csv_add2\datasets_fbt\merger\ISCX-VPN
# output_dir_train_test = os.path.join('..', 'TrafficData', 'datasets_csv_add2', 'datasets_fbt', 'merger', 'ISCX-VPN')
output_dir_blocks_first = os.path.join('..', 'TrafficData', 'datasets_fbt', 'truncation', 'dataset_20_d2')
# output_dir_blocks_second = os.path.join('..', 'TrafficData', 'datasets_fbt', 'augment', 'dataset_20_d2')
output_dir_blocks_second = os.path.join('..', 'TrafficData', 'datasets_csv_add2', 'datasets_fbt', 'merger', 'ISCX-TOR-Application')

In [3]:
global_stratified_split_memory_optimized(ori_csv_path, output_dir_train_test)

###   开始执行【内存优化版】的全局数据集分割   ###

[Pass 1/3] 正在扫描 ..\TrafficData\datasets_consolidate\dataset_20_d2.csv 以收集标签索引...


Scanning labels: 63it [00:15,  4.03it/s]


 -> 扫描完成。共 6256032 条记录，20 个唯一类别。

[Pass 2/3] 正在对索引进行分层抽样...


Splitting indices: 100%|██████████| 20/20 [00:02<00:00,  7.23it/s]



分割完成！各数据集规模如下:
 - 训练集 (Train Set): 5004812 条
 - 验证集 (Validation Set): 625610 条
 - 测试集 (Test Set): 625610 条

[Pass 3/3] 正在逐块读取并写入分割后的文件...


Writing split files: 63it [04:57,  4.72s/it]



文件写入完成！
 - 训练集已保存到: ..\TrafficData\datasets_split\dataset_20_d2\train_set.csv
 - 验证集已保存到: ..\TrafficData\datasets_split\dataset_20_d2\validation_set.csv
 - 测试集已保存到: ..\TrafficData\datasets_split\dataset_20_d2\test_set.csv

全局数据集分割步骤已成功完成！


In [5]:
train_set_path = os.path.join(output_dir_train_test, 'train_set.csv')

truncate_to_block_by_schema(train_set_path, output_dir_blocks_first)

Data loading completed, 5004812 records here. 
Fingerprints has been built. 
There are 228 Field Block in total. 


Saving blocks: 100%|██████████| 228/228 [01:08<00:00,  3.32it/s]


Truncation succeeded! 


In [6]:
merge_field_blocks_tree_similarity(output_dir_blocks_first, output_dir_blocks_second, similarity_threshold=0.8)

开始执行基于【结构化树相似度】的Field Block合并与剪枝流程...

[步骤 1/5] 正在扫描并分析所有原始Block...


Scanning blocks: 100%|██████████| 228/228 [00:04<00:00, 46.15it/s]



[步骤 2/5] 识别完成:
 - 44 个核心专家 (样本数 >= 1000)
 - 184 个待合并的小型专家

[步骤 3/5] 正在为小型专家寻找最佳合并目标...


Merging small blocks: 100%|██████████| 184/184 [00:05<00:00, 34.91it/s]



[步骤 4/5] 正在处理 6 个“孤儿”Block...

[步骤 5/5] 正在保存 45 个最终的Block...


Saving final blocks: 100%|██████████| 45/45 [00:31<00:00,  1.41it/s]



合并与剪枝成功！最终生成了 45 个专家Block。
合并日志已保存到: ..\TrafficData\datasets_fbt\augment\dataset_20_d2\merge_log.json


In [3]:
# main_block_name = '24'
# main_block_name = '34'
# output_path = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'train_test', 'blocks', 'chief_block_v2.csv')
# output_path = os.path.join('..', 'TrafficData', 'dataset_20_d2_csv_merged', 'train_test', 'blocks', 'chief_block_dataset_20_d2.csv')
output_path = os.path.join('..', 'TrafficData', 'datasets_csv_add2', 'datasets_final', 'ISCX-TOR-Application_chief_block_topk_augmented.csv.csv')
# augment_main_block(output_dir_blocks_second, main_block_name, output_path)
# augment_main_block_v2(output_dir_blocks_second, output_path)
augment_main_block_top_k(output_dir_blocks_second, output_path)

开始执行【Top-3合并】的数据补充策略...

[步骤 1/6] 正在扫描所有Block以确定Top-3 Chief Blocks...


Finding largest blocks: 100%|██████████| 40/40 [00:00<00:00, 9992.39it/s]

 -> 自动选定 Top-3 Block(s) 作为Chief Block：
    - '323' (文件大小: 18.66 MB)
    - '310' (文件大小: 13.32 MB)
    - '300' (文件大小: 11.32 MB)

[步骤 2/6] 正在加载并合并Top-3 Block(s)...


 -> “超级Chief Block”创建成功。
 -> 原始 Top-K 样本数: 124752
 -> 最终特征（节点）数: 81

[步骤 3/6] 扫描所有Block，建立类别分布情报库...


Scanning Blocks for labels: 100%|██████████| 40/40 [00:01<00:00, 31.55it/s]



[步骤 4/6] 在 “超级Chief Block” 中找到 1 个需要补充的类别:
['Google']

[步骤 5/6] 开始从所有其他Block中寻找并合并补充数据...


Augmenting Classes: 100%|██████████| 1/1 [00:00<00:00,  1.68it/s]



正在合并所有数据...

数据补充成功！
 - 原始 Chief Block ('['323', '310', '300']') 样本数: 124752
 - 补充后总样本数: 131199
 - 最终数据集已保存到: ..\TrafficData\datasets_csv_add2\datasets_final\ISCX-TOR-Application_chief_block_topk_augmented.csv.csv


In [3]:
main_block_name = '24'
output_path = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'train_test', 'blocks', 'specific_classes_intersec.csv')
create_specialist_dataset_intersect(output_dir_blocks_second, main_block_name, output_path, ['google', 'twitter'])

### 创建“共同独有特征专家模型”专用训练集 ###
### 目标类别: ['google', 'twitter'] ###

[步骤 1/5] 正在从所有Block中收集 google, twitter 的数据...


 -> 数据收集完成，共找到 20240 个相关样本。
 -> 数据来源于以下 6 个'捐献者'Block: {'44', '40', '39', '77', '35', '34'}

[步骤 2/5] 正在计算“共同的独有特征”...
 -> 计算完成，找到 6 个共同独有特征: ['tcp.option_kind', 'tcp.option_len', 'tcp.options.nop', 'tcp.options.timestamp', 'tcp.options.timestamp.tsecr', 'tcp.options.timestamp.tsval']

[步骤 3/5] 正在创建只包含独有特征的数据集...

[步骤 4/5] 正在保存最终的专家训练集...

专家训练集创建成功！
 - 总样本数: 20240
 - 总特征数: 6
 - 最终数据集已保存到: ..\TrafficData\dataset_29_d1_csv_merged\train_test\blocks\specific_classes_intersec.csv
